# Enrich Spotify Listening History with Metadata

The enriched data comes from this endpoint: https://developer.spotify.com/documentation/web-api/reference/get-track

The only field that looks interesting and needs further explanation is the `popularity` field. In the docs it says the following:

The popularity of the track. The value will be between 0 and 100, with 100 being the most popular.
The popularity of a track is a value between 0 and 100, with 100 being the most popular. The popularity is calculated by algorithm and is based, in the most part, on the total number of plays the track has had and how recent those plays are.
Generally speaking, songs that are being played a lot now will have a higher popularity than songs that were played a lot in the past. Duplicate tracks (e.g. the same track from a single and an album) are rated independently. Artist and album popularity is derived mathematically from track popularity. Note: the popularity value may lag actual popularity by a few days: the value is not updated in real time.

In [8]:
import pandas as pd

metadata = pd.read_parquet('../data/tracks_metadata.parquet')
print(metadata.loc[0])
print(metadata.info())

artists                         [{'external_urls': {'spotify': 'https://open.s...
available_markets                                                              []
disc_number                                                                     1
duration_ms                                                                240009
explicit                                                                    False
href                            https://api.spotify.com/v1/tracks/1TW9J0imyVBM...
id                                                         1TW9J0imyVBMihuwHqJ5sf
is_local                                                                    False
name                                                               Matter of Time
popularity                                                                      0
preview_url                                                                  None
track_number                                                                    1
type            

In [9]:
# I want to keep these keys: uri, duration_ms, explicit, popularity, external_urls.spotify
metadata = metadata[['uri', 'duration_ms', 'explicit', 'popularity', 'external_urls.spotify']]
metadata = metadata.rename(columns={'external_urls.spotify': 'spotify_url', 'duration_ms': 'length_ms'})
print(metadata.head())

                                    uri  length_ms  explicit  popularity  \
0  spotify:track:1TW9J0imyVBMihuwHqJ5sf     240009     False           0   
1  spotify:track:2NeSirLM2VHQW4upn0nMfB     233978     False           3   
2  spotify:track:1hgX0ZmmP7IRRjUFCIBrnQ     310133     False          63   
3  spotify:track:0Psz3az3RIYfJpnsajBT8N     190826     False          62   
4  spotify:track:439X8jGytErRiPnaoUJHju     218200     False          62   

                                         spotify_url  
0  https://open.spotify.com/track/1TW9J0imyVBMihu...  
1  https://open.spotify.com/track/2NeSirLM2VHQW4u...  
2  https://open.spotify.com/track/1hgX0ZmmP7IRRjU...  
3  https://open.spotify.com/track/0Psz3az3RIYfJpn...  
4  https://open.spotify.com/track/439X8jGytErRiPn...  


In [10]:
songs = pd.read_csv('../data/spotify.csv')
enriched = songs.merge(metadata, on='uri', how='left')
print(enriched.head())
enriched.to_csv('../data/spotify_enriched.csv', index=False)

            time_start             time_end  ms_played  \
0  2023-03-13 23:32:19  2023-03-13 23:32:47      27478   
1  2023-03-13 23:32:48  2023-03-13 23:36:42     233978   
2  2023-03-13 23:36:41  2023-03-13 23:41:52     310133   
3  2023-03-13 23:41:53  2023-03-13 23:45:04     190826   
4  2023-03-13 23:45:03  2023-03-13 23:48:42     218200   

                               track      artist  \
0                     Matter of Time    Vandelux   
1                Starry Night - Edit   Peggy Gou   
2  Feel Your Weight - Poolside Remix        Rhye   
3                               Nanã  Polo & Pan   
4                            CHANCES  KAYTRANADA   

                                    uri     id  length_ms  explicit  \
0  spotify:track:1TW9J0imyVBMihuwHqJ5sf  16308     240009     False   
1  spotify:track:2NeSirLM2VHQW4upn0nMfB  16309     233978     False   
2  spotify:track:1hgX0ZmmP7IRRjUFCIBrnQ  16310     310133     False   
3  spotify:track:0Psz3az3RIYfJpnsajBT8N  16311     190